# 01. HCG Schema — Treating a Causal Graph as an Object

Humm does not leave the causal graph to an LLM's internal representation; it stores it as an explicit object called the **Humm Causal Graph (HCG)**. Nodes and edges carry provenance, evidence, confidence and status, and this information feeds later validation (`humm.validate`) and training (`humm.gym`, `humm.fast`).

This notebook covers the following.

1. Build a graph from nodes, edges and evidence
2. Compute edge confidence (noisy-OR) and status (hypothesis → supported) from the evidence
3. Find cycles and mark the weakest edge as refuted
4. Serialize to LLM prompt text and save as JSON
5. Load a bnlearn ground-truth graph as an HCG and compute the SHD and SID metrics

In [ ]:
import networkx as nx
import pandas as pd

from humm.bench import datasets as D, metrics as M
from humm.paths import GRAPHS
from humm.graph import HCG, Evidence, EvidenceKind, to_compact_json, to_edge_text, to_prompt_block

## 1. Building the graph

In a marketing example, we build the structure `season → ad spend → visits → sales` plus `season → sales`. Season is a common cause of ad spend and sales, that is, a confounder. Each edge carries evidence, and the evidence kind (`text`, `data`, `llm_prior`, `expert`, `experiment`) determines its confidence weight.

In [ ]:
g = HCG(graph_id="sales-demo", assumptions=["계절 외의 숨은 공통 원인은 없다"])
for nid, label in [("season", "계절"), ("ad", "광고비"), ("visits", "방문"), ("sales", "매출")]:
    g.add_node(nid, label=label, dtype="continuous")

g.add_edge("season", "ad", sign="+", mechanism="성수기에 광고 예산을 늘린다",
           evidence=[Evidence(kind=EvidenceKind.EXPERT, score=0.9, ref="마케팅팀 인터뷰")])
g.add_edge("ad", "visits", sign="+", mechanism="노출이 늘면 방문이 는다",
           evidence=[Evidence(kind=EvidenceKind.TEXT, score=0.8, ref="report_07.pdf#p3"),
                     Evidence(kind=EvidenceKind.DATA, score=0.9, method="PC+BK")])
g.add_edge("visits", "sales", sign="+", mechanism="방문이 구매로 전환된다",
           evidence=[Evidence(kind=EvidenceKind.TEXT, score=0.7), Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.9)])
g.add_edge("season", "sales", sign="+", mechanism="성수기에는 수요 자체가 크다",
           evidence=[Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.6)])
g.is_dag()

## 2. Confidence and status

Edge confidence is the noisy-OR combination of the evidence, `1 − Π(1 − w_kind · score)`. With two or more different kinds of evidence, an edge is promoted from `hypothesis` to `supported`. The `verified` and `refuted` verdicts require data tests, so they are handled by `humm.validate` (see notebook 05).

In [ ]:
for e in g.edges:
    e.promote()
pd.DataFrame([{"edge": f"{e.src}→{e.dst}", "n_evidence": len(e.evidence), "kinds": sorted({x.kind.value for x in e.evidence}),
               "confidence": round(e.confidence, 3), "status": e.status.value} for e in g.edges])

## 3. Resolving cycles

When a reverse statement such as "higher sales lead to more ad spend" is extracted from text, a cycle appears. A cycle without a time lag breaks the DAG assumption, so for each cycle the edge with the lowest confidence is marked `refuted`. Refuted edges are kept as a record rather than deleted.

In [ ]:
g.add_edge("sales", "ad", sign="+", evidence=[Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.3)])
print("DAG:", g.is_dag(), "| cycles:", g.cycles())
print("refuted:", g.break_cycles(), "| DAG:", g.is_dag())

## 4. Serialization and saving

When giving a graph to an LLM, we use the edge list without refuted edges. `to_edge_text` is a human-readable format and `to_compact_json` is a token-saving format.

In [ ]:
print(to_prompt_block(g))
print()
print(to_edge_text(g, with_confidence=True))
print()
print(to_compact_json(g))

In [ ]:
path = g.save(GRAPHS / "sales-demo.json")
assert HCG.load(path) == g
path

## 5. Ground-truth graphs and metrics

We load the bnlearn Asia network, convert it to an HCG, and compare it with deliberately corrupted graphs. **SHD** counts structural differences (missing, extra, reversed edges), and **SID** counts the variable pairs whose interventional effect comes out wrong when computed with the incorrect graph. For causal inference, SID is the more direct metric.

In [ ]:
asia = HCG.from_networkx(D.load_bnlearn("asia"), "asia")
true = asia.to_networkx()

def edit(g, remove=(), add=()):
    h = g.copy()
    h.remove_edges_from(remove)
    h.add_edges_from(add)
    return h

variants = {
    "true graph": true,
    "smoke→lung reversed": edit(true, remove=[("smoke", "lung")], add=[("lung", "smoke")]),
    "either→dysp missing": edit(true, remove=[("either", "dysp")]),
    "empty graph": edit(true, remove=list(true.edges)),
}
pd.DataFrame({k: M.graph_report(true, v) for k, v in variants.items()}).T

A single reversal gives an SHD of 1, but the SID is much larger, because a wrong direction on one edge also breaks the adjustment sets of the many interventional effects that pass through it. Graphs produced by discovery algorithms can be evaluated with these metrics.

## Result files

Save an executed notebook under a different name than the original, and keep the original in `notebooks/` without outputs.

| File | Required | Notes |
|---|---|---|
| `data/graphs/sales-demo.json` | Optional | Fixed seed, reproducible |